# Predicción de diabetes tipo 2 a partir del microbioma intestinal humano: presentación de avance

**Proyecto Machine Learning**

**Objetivo de la presentación**:
- Mostrar la elección del tema
- Viabilidad del dataset
- Principales hallazgos del EDA
- Primeros modelos supervisados

## 1. Contexto y motivación

- El microbioma intestinal está relacionado con metabolismo, inflamación y salud humana.
- La diabetes tipo 2 (`t2d`) es una condición metabólica con posible relación con cambios en la composición microbiana.
- **El objetivo no es diagnosticar clínicamente**, sino explorar si el perfil taxonómico del microbioma contiene <u>señal útil para diferenciar controles y muestras asociadas a `t2d`</u>.

**Pregunta inicial:**

> ¿Puede el microbioma intestinal ayudar a distinguir muestras control (`n`) de muestras asociadas a diabetes tipo 2 (`t2d`)?


## 2. Dataset seleccionado

**Dataset:** Human Metagenomics - Kaggle  
**Archivo usado:** `abundance_stoolsubset.csv`

Tras limpieza inicial:

- 1577 muestras únicas.
- 2135 columnas totales.
- 2128 variables taxonómicas del microbioma.
- 7 columnas de metadata conservadas.
- Variable clínica principal: `disease`.

**Motivo de elección:**

- Cumple volumen mínimo del proyecto.
- Tiene complejidad real: alta dimensionalidad, *sparsity* y clases clínicas.
- Permite EDA, modelos supervisados, modelo no supervisado e interpretación biológica.


## 3. Limpieza realizada

Principales decisiones:

- Reducción de metadata a columnas interpretables.
- Normalización de valores faltantes y etiquetas.
- Unificación de etiquetas clínicas equivalentes:
  - `obese` -> `obesity`
  - `leaness` / `underweight` -> `leanness`
- Eliminación de duplicados por `sampleID`.
- Exclusión de muestras con etiquetas clínicas contradictorias.
- Sustitución de nulos categóricos en `gender` y `country` por `unknown`.

**Resultado:** dataset limpio sin `sampleID` duplicados y con `disease` completo.


## 4. Hallazgos principales del EDA

1. La variable `disease` está desbalanceada.
    - Las clases no tienen el mismo número de muestras (Fig 1. "*Distribución de muestras por condición clínica*"). 
        -  Esto obliga a usar métricas más allá del `accuracy` y a definir un `target` inicial acotado.
2. Muchas variables taxonómicas presentan alta *sparsity*: numerosos taxones aparecen como cero en gran parte de las muestras (Fig 2. "*Distribución del porcentaje de ceros por taxon*").
    - Esto aumenta ruido/dimensionalidad y justifica selección de variables o reducción dimensional.
3. Los heatmaps <u>agregados por *clase* bacteriana</u> muestran patrones relativos distintos entre condiciones clínicas (Fig 3. "*Patrones relativos de abundancia por clase bacteriana y condición clínica*").
    - Apoya a continuar con modelos predictivos.
4. El PCA <u>no muestra una separación clínica simple en PC1/PC2</u>. (Fig 4. "*PCA del microbioma coloreado por condición clínica - zoom central*").
    - La señal puede estar en combinaciones más complejas o componentes posteriores.
5. El PCA coloreado por `dataset_name` sugiere posible efecto de estudio de origen.

**Conclusión del EDA:**

> El dataset es viable, pero requiere **modelado cuidadoso** por alta dimensionalidad, sparsity, desbalance y posible *confounding* por origen de datos.


![Distribución de condiciones](../docs/figures/muestra_condicion.png)

![Sparsity taxones](../docs/figures/sparsity_taxones.png)

![Heatmap por enfermedad](../docs/figures/heatmap_clases_por_enfermedad.png)

![PCA microbioma zoom](../docs/figures/pca_microbioma_disease_zoom.png)

## 5. Definición del problema de Machine Learning

**Tipo de problema:** clasificación binaria supervisada.

**Target:**

- `0`: control (`n`)
- `1`: diabetes tipo 2 (`t2d`)

**Features iniciales:**

- Solo columnas taxonómicas que empiezan por `k__`.

**Variables no usadas inicialmente como predictores:**

- `sampleID`: identificador.
- `dataset_name`: posible sesgo por estudio de origen.
- `country`: posible sesgo geográfico.
- `age`, `gender`, `bmi`: variables de contexto/EDA, no usadas en el modelo base.

**Objetivo:** evaluar si las abundancias microbianas contienen señal predictiva por sí mismas.


## 6. Pipeline de modelado

Se usaron pipelines para evitar fuga de información.

**Flujo general**:

1. `train_test_split` estratificado.
2. Transformación `log1p` de abundancias.
3. Selección de 300 features con `SelectKBest`.
4. Escalado en modelos sensibles a escala.
5. Entrenamiento del clasificador.

**Nota:**

> Las transformaciones se ajustan únicamente con el **conjunto de entrenamiento** y luego se aplican al **conjunto de test**.

## 7. Modelos supervisados probados

Se entrenaron <u>cinco modelos supervisados</u>:

1. Logistic Regression
2. Random Forest
3. Gradient Boosting
4. SVM
5. KNN

**Rol de los modelos:**

- Logistic Regression: baseline interpretable.
- Gradient Boosting: modelo no lineal con mayor capacidad global de separación.
- Random Forest / SVM / KNN: comparación de enfoques alternativos.


## 8. Resultados comparativos

Tabla 1. Resultados comparativos
| **Modelo** | **Accuracy** | **Precision** | **Recall** | **F1-score** | **ROC-AUC** |
|---|---:|---:|---:|---:|---:|
| Gradient Boosting | 0.765 | 0.514 | 0.422 | 0.463 | 0.792 |
| Random Forest | 0.759 | 0.500 | 0.111 | 0.182 | 0.783 |
| SVM | 0.717 | 0.435 | 0.600 | 0.505 | 0.766 |
| Logistic Regression | 0.759 | 0.500 | 0.622 | 0.554 | 0.745 |
| KNN | 0.749 | 0.429 | 0.133 | 0.203 | 0.632 |

**Lectura principal:**

1. Gradient Boosting obtiene el **mejor ROC-AUC** = 0.792 (Fig 5. "*Comparación de modelos supervisados - ROC-AUC*", Fig 7. "*Curvas ROC comparativas*").
2. Logistic Regression obtiene mejor `recall` y `F1-score` para `t2d`. (Tabla 1. "*Resultados comparativos*").
3. Existe un *trade-off* entre **separación global** y **detección** de la clase minoritaria.
    - Los modelos no están ganando en todo al mismo tiempo. Algunos separan mejor las clases en general, pero *detectan peor* la clase minoritaria `t2d`.

![comparacion modelos roc](../docs/figures/comparacion_modelos_roc_auc.png)

![confusion matrices](../docs/figures/confusion_matrices_gradient_logistic.png)

- Gradient Boosting detecta 19 de 45 `t2d`.
- Logistic Regression detecta 28 de 45 `t2d`.

**Trade-off**
- Gradient Boosting: mejor separación global pero detecta menos `t2d` con el umbral actual.
- Logistic Regression: peor ROC-AUC pero detecta más `t2d`.

Gradient Boosting separa mejor globalmente; Logistic Regression detecta más t2d. La elección depende de si priorizamos **ranking global** o **sensibilidad**.

![confusion matrices](../docs/figures/curvas_roc_comparativas.png)

## 9. Interpretación de resultados

**Gradient Boosting**

- Mejor ROC-AUC: 0.79.
- Mejor capacidad global de separación entre `n` y `t2d`.
- Detecta menos casos reales de `t2d` con el umbral por defecto.

**Logistic Regression**

- Baseline interpretable.
- Mejor recall para `t2d`: detecta 28 de 45 casos reales.
- Mayor número de falsos positivos.

**Conclusión:**

> El microbioma muestra <u>señal predictiva moderada</u> para diferenciar controles y diabetes tipo 2. **Gradient Boosting** destaca por separación global, mientras **Logistic Regression** es útil por sensibilidad e interpretabilidad.


## 10. Limitaciones actuales

- Modelos entrenados con configuración inicial, <u>sin hiperparametrización</u>.
- Posible efecto de dataset de origen.
- Clases desbalanceadas.
- Alta dimensionalidad y sparsity.
- El target `n` debe interpretarse como control del dataset, no necesariamente salud metabólica perfecta.
- **El modelo no debe interpretarse como herramienta diagnóstica**.


## 11. Próximos pasos...

1. Optimización de hiperparámetros.
2. Ajuste de umbral de decisión.
3. Interpretabilidad de variables/taxones importantes.
4. Modelo no supervisado: PCA + KMeans o clustering jerárquico.
5. Evaluación más robusta.
6. Selección del modelo final.
7. Desarrollo de demo en *Streamlit*.

**Conclusiones:**

> El primer modelado confirma que <u>existe señal útil en el microbioma</u>, pero aún se requiere **optimización** e **interpretación** antes de seleccionar un modelo final.